In [ ]:
!pip install -q ufal.chu_liu_edmonds

In [ ]:
from collections import defaultdict
from itertools import pairwise
import json
import re
from string import punctuation

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

In [ ]:
%cd ..

In [ ]:
results_gold = defaultdict(list)
target_senses = defaultdict(set)
examples = []
with open("drive/MyDrive/dev_dpo.jsonl", "r") as f:
    for i, line in enumerate(f):
        line = json.loads(line)
        examples.append(line["example"])

        results_gold[line["target"]].append(line["example"])
        target_senses[line["target"]].add(line["definition"])

In [ ]:
!curl -sSfL https://hf.co/git-xet/install.sh | sh
!git clone https://huggingface.co/HPLT/hplt_gpt_bert_base_3_0_eng_Latn-UD

In [ ]:
!pip install transformers==4.57.6 -q

In [ ]:
!pwd

In [ ]:
%cd hplt_gpt_bert_base_3_0_eng_Latn-UD/

In [ ]:
from lemma_rule import apply_lemma_rule
from preprocessor import Preprocessor
from utils import load_model

In [ ]:
class SentencesDataset(Dataset):
    def __init__(self, sentences):
        self.sentences = sentences

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return self.sentences[idx]


device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer, model, predict_model = load_model(model_path="HPLT/hplt_gpt_bert_base_3_0_eng_Latn", device=device)
predict_model.eval()
preprocessor = Preprocessor(tokenizer=tokenizer)
dataset = SentencesDataset(examples)


In [ ]:
from copy import copy
dataloader = DataLoader(dataset, batch_size=64, num_workers=2)

lemmas = []
for batch in tqdm(dataloader):
    orig = copy(batch)
    batch = preprocessor.preprocess(batch)
    with torch.no_grad():
        try:
            lemma_p, _, _, _, _, _, _, _ = predict_model(
                                    batch["subwords"].to(device),
                                    batch["alignment"].to(device),
                                    batch["subword_lengths"].to(device),
                                    batch["word_lengths"].to(device),
                                    )
        except AssertionError as e:
            print(orig)
    for i in range(len(batch["words"])):
        sentence_lemmas = []
        for j, form in enumerate(batch["words"][i]):
            lemma_rule = {
                rule_type: model["dataset"].lemma_vocab[rule_type].get(p[i, j, :].argmax().item(),
                        model["dataset"].lemma_vocab[rule_type][-1])
                for rule_type, p in lemma_p.items()
            }
            lemma = apply_lemma_rule(form, lemma_rule)
            sentence_lemmas.append(lemma)
        lemmas.append(" ".join(sentence_lemmas)+'\n')

with open("../drive/MyDrive/dev_lemmas.txt", "w") as f:
    f.writelines(lemmas)

In [ ]:
def preprocess(x, target, len_target):
    lemmatized, splitted = x.lemmas, x.spl
    idx = lemmatized.lower().find(target.lower())
    word_idx = len(lemmatized[:idx].split(" ")) - 1
    len_target = len(target.split(" "))
    splitted = re.sub(r"\s+", " ", splitted).split(" ")
    result = " ".join(splitted[:word_idx] + ["<t>" + " ".join(splitted[word_idx: word_idx+len_target]) + " </t>"] + splitted[word_idx + len_target:])
    return result

In [ ]:
with open("drive/MyDrive/dev_lemmas.txt", "r") as f:
    lemmas = f.readlines()
lemmas[0]

In [ ]:
punct_pattern = re.compile(r"([{}])".format(re.escape(punctuation)))

In [ ]:
results_gold = defaultdict(list)
target_senses = defaultdict(set)
with open("drive/MyDrive/dev_dpo.jsonl", "r") as f:
    for lemma, line in tqdm(zip(lemmas, f)):
        line = json.loads(line)
        spl = re.sub(punct_pattern, r" \1 ", line["example"]).strip()
        spl = re.sub(r'\s+', ' ', spl)
        lemma = lemma.rstrip()
        proc = preprocess(lemma, spl, line["target"], len(line["target"]))
        results_gold[line["target"]].append(proc)
        target_senses[line["target"]].add(line["definition"])
        #if len(line["target"].split()) > 1:
        #    print(proc)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sachinn1/xl-durel", processor_kwargs={"truncation": False}, device=device)

In [ ]:
green = pd.read_csv("drive/MyDrive/green-periti.tsv", sep="\t")
green.head()

In [ ]:
green["spl"] = green.example.apply(lambda x: re.sub(punct_pattern, r" \1 ", x).strip())
green.head()

In [ ]:
model.eval()
table_2 = defaultdict(list)
for target, t_result in tqdm(green.groupby("target")):
    print(target)
    len_target = len(target)
    year_predictions = []
    for i, (year, y_result) in enumerate(t_result.groupby("year")):
        y_result["processed"] = y_result.apply(preprocess, args=(target, len_target), axis=1)
        usage_matrix2 = model.encode(y_result["processed"].values)
        print(usage_matrix2.shape)
        if i > 0:
            for year_prediction in year_predictions:
                usage_matrix1 = year_prediction[1]
                samples_1 = usage_matrix1.shape[0]
                samples_2 = usage_matrix2.shape[0]
                min_samples = min(samples_1, samples_2)
                usage_matrix1 = usage_matrix1[:min_samples, :]
                usage_matrix_2 = usage_matrix2[:min_samples, :]

                sim = np.mean(cosine_similarity(usage_matrix1, usage_matrix_2))
                table_2["target"].append(target)
                table_2["years"].append(f"{year_prediction[0]}, {year}")
                apd = 1 - float(sim)
                table_2["apd"].append(apd)
        year_predictions.append((year, usage_matrix2))
pd.DataFrame(table_2).to_csv("drive/MyDrive/apd-xld-eu.tsv", sep="\t", index=False)
